# Can we predict next-month churn?
Target: `churn_next_month`. Data: monthly account snapshots (one row per account-month).

Plan: inspect data → check for leakage → honest validation (time-based split, no account in both train and test) → compare to baselines → interpret.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv')
print(df.shape, df.account_ref.nunique(), 'accounts;', df.snapshot_month.min(), '->', df.snapshot_month.max())
print(df.isna().sum().sum(), 'missing;', df.duplicated(['account_ref','snapshot_month']).sum(), 'duplicate account-months')
print('churn rate:', df.churn_next_month.mean().round(3))
df.describe().T

## Leakage check: `lifetime_months`
The dictionary says it is total months active *from a CRM export of 2026-01-05* — i.e. it's computed using information after the snapshot. Compare it with `tenure_months`.

In [ ]:
gap = df.lifetime_months - df.tenure_months
print(gap.groupby(df.churn_next_month).describe())

For every churned row `lifetime_months == tenure_months` (gap is exactly 0), while for non-churners it's mostly larger. The column encodes the outcome, so it **is excluded**. It would not be available at prediction time, and including it gives a spuriously great score. Shown below, then dropped.

In [ ]:
num = ['seats','monthly_fee','tenure_months','usage_hours','support_tickets']
cat = ['plan']
def pipe(model, nums):
    pre = ColumnTransformer([('n', StandardScaler(), nums), ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
    return make_pipeline(pre, model)
df['month'] = pd.PeriodIndex(df.snapshot_month, freq='M')
cut = pd.Period('2025-06','M')
train, test = df[df.month < cut], df[df.month >= cut]
# drop test rows of accounts seen in training, so no account is in both sets
test = test[~test.account_ref.isin(train.account_ref)]
print(len(train), len(test), 'test churn rate', test.churn_next_month.mean().round(3))
leaky = pipe(LogisticRegression(max_iter=1000), num+['lifetime_months']).fit(train, train.churn_next_month)
print('WITH leaky lifetime_months  AUC:', round(roc_auc_score(test.churn_next_month, leaky.predict_proba(test)[:,1]), 3))

## Calendar effect
Churn rate swings from ~3% to ~55% by month. That is a regime shift that snapshot features may not explain, so `snapshot_month` is not used as a feature: future months can't be seen at deployment.

In [ ]:
m = df.groupby('month').churn_next_month.mean()
ax = m.plot(marker='o', figsize=(9,3), color='#2a6f97'); ax.set_ylabel('churn rate'); ax.set_title('Next-month churn rate by snapshot month'); plt.show()

In [ ]:
fig, axs = plt.subplots(1,3, figsize=(13,3))
for ax, col in zip(axs, ['usage_hours','support_tickets','tenure_months']):
    g = pd.cut(df[col], 6) if col=='usage_hours' else df[col].clip(upper=12)
    df.groupby(g, observed=True).churn_next_month.mean().plot(ax=ax, marker='o', color='#2a6f97')
    ax.set_title(f'churn rate vs {col}'); ax.tick_params(axis='x', labelsize=7)
plt.tight_layout(); plt.show()
print(df.groupby('plan').churn_next_month.mean().round(3))

## Honest evaluation (no `lifetime_months`)
Train on snapshots before 2025-06, test on 2025-06 onward for **accounts not seen in training**. Baseline = base rate (AUC 0.5).

In [ ]:
y_tr, y_te = train.churn_next_month, test.churn_next_month
models = {'logistic': pipe(LogisticRegression(max_iter=1000), num),
          'grad. boosting': pipe(HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0), num)}
rows=[]
for n,mdl in models.items():
    mdl.fit(train, y_tr); p = mdl.predict_proba(test)[:,1]
    top = np.argsort(-p)[:int(.1*len(p))]
    rows.append({'model':n,'AUC':roc_auc_score(y_te,p),'PR-AUC':average_precision_score(y_te,p),
                 'precision@top10%':y_te.values[top].mean(),'lift@top10%':y_te.values[top].mean()/y_te.mean()})
rows.append({'model':'base rate','AUC':.5,'PR-AUC':y_te.mean(),'precision@top10%':y_te.mean(),'lift@top10%':1})
pd.DataFrame(rows).set_index('model').round(3)

## What drives it?

In [ ]:
best = models['grad. boosting']
r = permutation_importance(best, test[num+cat], y_te, scoring='roc_auc', n_repeats=10, random_state=0)
imp = pd.Series(r.importances_mean, index=num+cat).sort_values()
imp.plot.barh(color='#2a6f97', figsize=(6,3), title='Permutation importance (drop in test AUC)'); plt.show()
lr = models['logistic']; names = lr[0].get_feature_names_out()
print(pd.Series(lr[-1].coef_[0], index=names).round(2).sort_values())

## Conclusions
- **Yes, moderately.** On held-out later months and unseen accounts, both models reach AUC ≈ 0.71–0.72 (0.5 = chance). The top-10% highest-risk accounts churn ~31% of the time versus a ~10% base rate (≈3x lift). Boosting does no better than logistic regression, so the signal is simple.
- **Drivers:** lower usage, more support tickets and longer tenure go with higher churn risk (logistic coefficients on standardized inputs). Plan type and seat count add little.
- **Leakage:** `lifetime_months` gives AUC 1.0 because it equals `tenure_months` for every churned row (it comes from a later CRM export). It must not be used.
- **Caveats:** churn rate swings from 3% to 55% by calendar month, which these features don't explain, so expect drift and re-check calibration before treating scores as probabilities. The test is a single temporal split (2,262 rows), so metrics are approximate. Next steps would be per-account history features (usage trend) and rolling-origin validation.